# Phase 3: Neural Network Building Blocks

## What you'll learn
- `nn.Module` — the base class for all models
- Common layers: Linear, Conv2d, BatchNorm, Dropout
- Activation functions
- `nn.Sequential` vs custom forward
- Inspecting parameters
- Weight initialization

---

In [ ]:
import torch
import torch.nn as nn

## 3.1 nn.Module — The Foundation

Every neural network in PyTorch inherits from `nn.Module`. You define:
1. **`__init__`** — declare layers
2. **`forward`** — define how data flows through layers

PyTorch handles the backward pass automatically via autograd.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleNet(784, 128, 10)
print(model)

# Test with dummy input
dummy = torch.randn(4, 784)  # batch of 4
output = model(dummy)
print(f"\nOutput shape: {output.shape}")  # (4, 10)

## 3.2 nn.Linear — Fully Connected Layer

Applies `y = xW^T + b`. The most basic building block.

In [ ]:
layer = nn.Linear(in_features=5, out_features=3)

print(f"Weight shape: {layer.weight.shape}")  # (3, 5)
print(f"Bias shape: {layer.bias.shape}")       # (3,)

x = torch.randn(2, 5)  # batch=2, features=5
out = layer(x)
print(f"Output shape: {out.shape}")  # (2, 3)

# Without bias
layer_no_bias = nn.Linear(5, 3, bias=False)
print(f"Has bias: {layer_no_bias.bias}")  # None

## 3.3 Activation Functions

Activations add **non-linearity** — without them, stacking linear layers is just one big linear layer.

| Function | Use Case | Range |
|----------|----------|-------|
| ReLU | Default hidden layers | [0, ∞) |
| GELU | Transformers | (-0.17, ∞) |
| Sigmoid | Binary output | (0, 1) |
| Tanh | Centered output | (-1, 1) |
| Softmax | Multi-class probabilities | (0, 1), sums to 1 |

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

# Functional API (torch.nn.functional or torch.*)
print(f"ReLU:    {torch.relu(x)}")
print(f"Sigmoid: {torch.sigmoid(x)}")
print(f"Tanh:    {torch.tanh(x)}")

# Module API (used inside nn.Module)
relu = nn.ReLU()
gelu = nn.GELU()
print(f"\nReLU module: {relu(x)}")
print(f"GELU module: {gelu(x)}")

# Softmax (along a dimension)
logits = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(logits, dim=0)
print(f"\nSoftmax: {probs}")
print(f"Sum: {probs.sum()}")  # 1.0

## 3.4 Common Layers

### Conv2d — Convolutional Layer (for images)

In [ ]:
# Conv2d(in_channels, out_channels, kernel_size)
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)

# Input: (batch, channels, height, width)
img = torch.randn(1, 3, 32, 32)
out = conv(img)
print(f"Conv2d: {img.shape} → {out.shape}")  # (1, 16, 32, 32)

### BatchNorm — Normalizes activations (stabilizes training)

In [ ]:
# BatchNorm1d for FC layers, BatchNorm2d for conv layers
bn1d = nn.BatchNorm1d(128)
bn2d = nn.BatchNorm2d(16)

x_fc = torch.randn(32, 128)       # batch=32, features=128
x_conv = torch.randn(32, 16, 8, 8) # batch=32, channels=16

print(f"BN1d: {bn1d(x_fc).shape}")
print(f"BN2d: {bn2d(x_conv).shape}")

### Dropout — Randomly zeros elements during training (prevents overfitting)

In [ ]:
dropout = nn.Dropout(p=0.5)  # 50% of neurons dropped

x = torch.ones(1, 10)

# Training mode — dropout active
dropout.train()
print(f"Train mode: {dropout(x)}")

# Eval mode — dropout disabled
dropout.eval()
print(f"Eval mode:  {dropout(x)}")

## 3.5 nn.Sequential — Quick Model Building

For simple feed-forward architectures, `nn.Sequential` saves boilerplate.

In [ ]:
# Instead of writing a full class:
model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

print(model)
out = model(torch.randn(4, 784))
print(f"\nOutput: {out.shape}")

### When to use Sequential vs custom Module?

| Use `nn.Sequential` | Use custom `nn.Module` |
|---------------------|------------------------|
| Simple linear stack | Skip connections (ResNet) |
| Quick prototyping | Multiple inputs/outputs |
| No branching logic | Conditional logic in forward |

## 3.6 Inspecting Parameters

Understanding what's inside your model is essential for debugging and optimization.

In [ ]:
model = SimpleNet(784, 128, 10)

# Count total parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,}")
print(f"Trainable:    {trainable:,}")

# Named parameters
print("\n--- Named Parameters ---")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

# Named modules
print("\n--- Named Modules ---")
for name, module in model.named_modules():
    print(f"{name}: {module.__class__.__name__}")

## 3.7 Weight Initialization

Good initialization helps training converge faster. PyTorch layers have reasonable defaults, but you can customize.

In [ ]:
def init_weights(m):
    """Custom weight initialization function."""
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model = SimpleNet(784, 128, 10)

# Apply to all layers
model.apply(init_weights)

# Verify
print(f"fc1 weight mean: {model.fc1.weight.mean():.4f}")
print(f"fc1 weight std:  {model.fc1.weight.std():.4f}")
print(f"fc1 bias:        {model.fc1.bias[:5]}")

# Common init methods:
# nn.init.xavier_uniform_()   — good for sigmoid/tanh
# nn.init.kaiming_normal_()   — good for ReLU
# nn.init.zeros_()            — for biases
# nn.init.constant_(tensor, val)

## 3.8 Freezing Layers

Useful for transfer learning — freeze pretrained layers and only train new ones.

In [ ]:
model = SimpleNet(784, 128, 10)

# Freeze fc1
for param in model.fc1.parameters():
    param.requires_grad = False

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params after freezing: {trainable:,}")

## ✅ Phase 3 Summary

You now know how to:
- Build models with `nn.Module` (define `__init__` + `forward`)
- Use common layers: Linear, Conv2d, BatchNorm, Dropout
- Choose activation functions for different scenarios
- Use `nn.Sequential` for simple architectures
- Inspect and count model parameters
- Initialize weights properly
- Freeze layers for transfer learning

**Next → Phase 4: Data Pipeline (Dataset & DataLoader)**